# [LEC] LangGraph 심화

슬라이드와 설명, 퀴즈로 그래프를 나누고(Subgraph), 상태를 저장하고(Persistence), 여러 작업을 동시에 돌리는(Parallel Execution) 방법을 익힙니다.

셀을 위에서부터 차례로 실행하면 슬라이드와 퀴즈가 나타납니다.

In [ ]:
%cd ../../../course/Chapter02/notebooks

**목차**

- 1. Subgraph (슬라이드) (퀴즈)
- 2. Persistence (슬라이드) (퀴즈)
  - 2.1 Checkpointer (슬라이드) (퀴즈)
  - 2.2 Memory Store (슬라이드) (퀴즈)
- 3. Parallel Execution (슬라이드) (퀴즈)
  - 3.1 Fan-out, Fan-in (슬라이드) (퀴즈)
  - 3.2 Send (슬라이드) (퀴즈)
- 4. 정리 퀴즈 (퀴즈)

In [ ]:
import sys

sys.path.insert(0, "..")  # 챕터 폴더의 assets 도구를 불러오기 위한 경로

from assets.widgets import show_quiz, show_slides, slide_setup

slide_setup()  # 슬라이드 뷰어(PDF.js)를 불러옵니다. 슬라이드가 비어 보이면 이 셀을 다시 실행합니다.

In [ ]:
show_slides("3-4", [1, 2])

이번 시간은 LangGraph로 조금 더 큰 그래프를 만드는 방법 세 가지를 봅니다.

1. Subgraph: 그래프를 기능 단위로 나누고, 나눈 그래프를 다른 그래프 안에서 노드처럼 씁니다.
2. Persistence: 실행 중의 상태를 저장합니다. 대화 하나를 이어 가는 Checkpointer와, 대화를 넘어 남기는 Memory Store로 나뉩니다.
3. Parallel Execution: 서로 기다릴 필요 없는 작업을 동시에 실행하고 결과를 합칩니다. 노드 수가 정해진 Fan-out/Fan-in과, 실행할 때 작업 수를 정하는 Send가 있습니다.

**왜 배우나:** 노드와 엣지, 조건부 엣지만으로도 Agent를 만들 수 있습니다. 하지만 흐름이 길어지고, 대화가 여러 번 이어지고, 처리할 일이 많아지면 그래프 하나에 모두 담기 어렵습니다. 이 세 가지는 그때 쓰는 도구입니다.

<details><summary>강사용 메모</summary>

| 항목 | 내용 |
|---|---|
| 버전 | C (슬라이드 요약 + 내용 있는 곳만 "왜 배우나") |
| 원본 PDF | `course/Chapter02/slides/day01/3-4-langgraph-advanced.pdf` (10쪽. 표지 p1, 목차 p2, 장 표지 p3·p5·p8, 본문 p4·p6·p7·p9·p10) |
| 원본 코드 | 이 노트북에는 없습니다. `course/Chapter02/code/raw/03-langgraph-advanced/`의 네 파일은 `3-4_langgraph_advanced_LAB.ipynb`에서 다룹니다. |
| API 호출 | 없습니다. 슬라이드 준비 셀만 실행하면 됩니다. |
| 시간 | 커리큘럼 3일차 3.4 "LangGraph 심화" 3.0H. 목차 단계에서 강의 약 40분 + 실습 약 120분 + 여유 20분으로 가정했습니다. |

---

- 앞 과정 근거: M1 2-1(LangGraph)에서 `StateGraph`·노드/엣지·`compile`·조건부 엣지·`MemorySaver`+`thread_id`·`add_messages` Reducer를 다뤘습니다.
- 이 권에서 처음 나오는 API: `SqliteSaver`, `get_state_history`(이력·time-travel), `InMemoryStore`(`put`/`search`, `compile(store=)`), `Send`.
- 슬라이드 본문이 5쪽뿐입니다. 쪽마다 코드가 실습 코드와 거의 같으므로, 설명할 때 코드 줄을 짚어 주면 실습으로 자연스럽게 넘어갑니다.

</details>

## 1. Subgraph <small>(슬라이드) (퀴즈)</small>

In [ ]:
show_slides("3-4", [3, 4])

Subgraph는 독립적으로 컴파일한 그래프를 상위 그래프의 노드처럼 쓰는 구조입니다.

- 복잡한 워크플로우를 기능 단위로 나눌 수 있습니다.
- Subgraph마다 State를 따로 정의할 수 있습니다.
- 요약, 번역, 문서 처리처럼 반복되는 흐름을 재사용하기 좋습니다.

연결하는 방법은 간단합니다. 부모 그래프의 노드 함수 안에서 컴파일된 자식 그래프를 `invoke`로 호출합니다. 슬라이드 코드에서 `summarize_document`는 부모의 `document`를 자식 State의 `text`에 넣어 `summary_subgraph`를 호출하고, 자식이 돌려준 `summary`를 부모 State에 담습니다. 부모와 자식의 State가 다르므로, 넘기고 받는 값을 노드 함수가 옮겨 줍니다.

**왜 배우나:** 그래프가 커지면 한 덩어리로는 읽기도 고치기도 어렵습니다. 요약처럼 여러 곳에서 되풀이되는 흐름은 한 번 만들어 두고 노드처럼 가져다 씁니다.

In [ ]:
show_quiz("../assets/quizzes/3-4/q1.json")

**정리:** 자식 그래프를 따로 컴파일하고, 부모 노드 함수 안에서 호출합니다. 부모와 자식의 State가 달라도 노드 함수가 값을 옮겨 줍니다.

## 2. Persistence <small>(슬라이드) (퀴즈)</small>

### 2.1 Checkpointer <small>(슬라이드) (퀴즈)</small>

In [ ]:
show_slides("3-4", [5, 6])

Checkpointer는 LangGraph 실행 중의 State를 저장하고, 같은 thread에서 이어서 실행할 수 있게 하는 장치입니다.

- `MemorySaver`: 프로세스 메모리에 저장하는 개발·데모용 체크포인터입니다.
- `SqliteSaver`: SQLite 파일에 저장해 재시작한 뒤에도 상태가 남습니다.
- `thread_id`로 사용자별, 세션별 실행 흐름을 구분합니다.

체크포인터를 쓰면 같은 `thread_id`의 이전 State를 이어받습니다. 슬라이드 코드에서 첫 호출에 "내 이름은 홍길동이야."를 보내고, 같은 `config`로 "내 이름이 뭐야?"를 보내면 앞 메시지를 이어받아 답합니다. 실행 이력을 조회하거나, 과거 체크포인트에서 다시 실행할 수도 있습니다.

**왜 배우나:** `MemorySaver`는 프로그램이 끝나면 기록이 사라집니다. 기록을 파일에 남기는 방법과, 지난 시점으로 되돌아가 다시 실행하는 방법을 알면 대화형 Agent를 멈췄다 이어 가거나 잘못된 흐름을 되짚어 볼 수 있습니다.

In [ ]:
show_quiz("../assets/quizzes/3-4/q21.json")

**정리:** Checkpointer는 같은 `thread_id`의 State를 이어 줍니다. 메모리에 두면(MemorySaver) 끝나면 사라지고, 파일에 두면(SqliteSaver) 남습니다.

<details><summary>강사용 메모</summary>

- "`MemorySaver`는 1·2일차에 썼다"는 M1 2-1 실습과 커리큘럼 2.2 근거입니다. 수강생 글에는 다른 노트북을 언급하지 않도록 쓰지 않았습니다. 말로 짚어 주면 됩니다.
- 슬라이드에는 `SqliteSaver`·이력 조회·time-travel 코드가 없습니다. 실습 코드(`02_persistence.py`)에 있습니다.
- 원본 코드 주석에는 실제 서비스용 `PostgresSaver`(`langgraph-checkpoint-postgres` 패키지)도 적혀 있습니다. 슬라이드에는 없습니다.

</details>

### 2.2 Memory Store <small>(슬라이드) (퀴즈)</small>

In [ ]:
show_slides("3-4", [7])

Memory Store는 Checkpointer와 별도로 사용자별 장기 기억을 저장하는 공간입니다.

- thread를 넘어 공유되는 장기 정보를 저장합니다.
- `namespace`로 사용자, 조직, 프로젝트별 기억을 나눕니다.
- 사용자 선호, 반복 업무 규칙, 중요한 사실을 저장하는 데 알맞습니다.

슬라이드 코드는 `InMemoryStore()`로 저장소를 만들고, `compile(checkpointer=MemorySaver(), store=store)`로 체크포인터와 함께 그래프에 넘깁니다. `("memories", user_id)` 같은 튜플이 namespace입니다. `store.search(namespace)`로 그 사용자의 기억을 찾고, `store.put(namespace, "fact_0", {...})`로 새 기억을 저장합니다.

**왜 배우나:** Checkpointer는 thread 단위입니다. 새 대화(새 thread)를 열면 앞 대화에서 들은 이름이나 취미를 모릅니다. 대화가 바뀌어도 남겨 둘 정보는 따로 저장해야 합니다.

In [ ]:
show_quiz("../assets/quizzes/3-4/q22.json")

**정리:** 한 대화를 이어 가는 것은 Checkpointer, 대화를 넘어 남길 사용자 정보는 Memory Store가 맡습니다. namespace로 누구의 기억인지 나눕니다.

<details><summary>강사용 메모</summary>

- 실습 원본 `03_memory_store.py`의 compile 줄은 `builder.compile()`로 checkpointer·store가 빠져 있습니다(루트 AGENTS.md 7절 알려진 결함). 실습 노트북에서는 슬라이드와 같은 줄로 바꿔 넣었습니다(원본 파일은 그대로).
- 그런데 실습 코드의 노드는 전역 변수 `store`를 직접 씁니다. 그래서 compile에 `store=`를 넘기든 안 넘기든 화면이 똑같습니다. LangGraph는 노드 함수가 `store` 인자를 선언할 때만 compile에 넘긴 store를 넣어 줍니다(설치 버전 1.2.12에서 확인). 수강생이 물으면 이 차이를 짚어 줍니다.

</details>

## 3. Parallel Execution <small>(슬라이드) (퀴즈)</small>

### 3.1 Fan-out, Fan-in <small>(슬라이드) (퀴즈)</small>

In [ ]:
show_slides("3-4", [8, 9])

Fan-out/Fan-in은 하나의 입력을 여러 작업으로 나눠 병렬로 실행한 뒤, 결과를 다시 합치는 패턴입니다.

- Fan-out: 독립적인 노드나 Worker로 작업을 나눠 보냅니다.
- Fan-in: 각 분기의 결과가 모이면 다음 노드에서 통합합니다.
- 장점·단점 분석, 여러 문서 요약, 다중 검색처럼 서로 독립적인 작업에 알맞습니다.
- 병렬 결과를 합치려면 State의 병합 규칙을 분명히 설계해야 합니다.

슬라이드 코드에서 `START`에서 `pros`와 `cons`로 엣지를 두 개 내면 두 노드가 함께 시작합니다(fan-out). `pros`와 `cons`에서 `summary`로 엣지를 모으면, 두 노드가 모두 끝난 뒤에 `summary`가 실행됩니다(fan-in).

**왜 배우나:** 서로 기다릴 필요 없는 작업을 차례로 돌리면 시간만 늘어납니다. 반대로 앞 단계의 결과가 있어야 하는 작업은 병렬로 나눌 수 없습니다. 어떤 작업이 독립적인지 가려내는 것이 먼저입니다.

In [ ]:
show_quiz("../assets/quizzes/3-4/q31.json")

**정리:** 엣지를 여러 개 내면 fan-out, 한 노드로 모으면 fan-in입니다. 서로 독립적인 작업만 나누고, 합칠 때의 병합 규칙을 정해 둡니다.

### 3.2 Send <small>(슬라이드) (퀴즈)</small>

In [ ]:
show_slides("3-4", [10])

Send는 실행 시점에 필요한 Worker 수를 정하는 LangGraph의 병렬 실행 API입니다.

- `Send("worker", state)` 모양으로 특정 노드에 별도 입력을 보냅니다.
- 조건부 Edge에서 `list[Send]`를 반환하면 Worker가 병렬로 생깁니다.
- 입력 리스트 길이에 따라 작업 수가 달라지는 상황에 쓸모가 있습니다.
- 결과를 누적하려면 `Annotated` reducer 같은 병합 전략이 필요합니다.

슬라이드 코드의 `fan_out`은 `queries`에 든 질문마다 `Send("worker", {"query": q, ...})`를 하나씩 만들어 목록으로 돌려줍니다. 이 함수를 `add_conditional_edges(START, fan_out, ["worker"])`에 넣으면, 질문이 세 개면 worker가 세 번, 다섯 개면 다섯 번 병렬로 실행됩니다.

**왜 배우나:** 질문이 몇 개 들어올지 그래프를 만들 때는 모를 수 있습니다. 엣지로 만든 fan-out은 노드 수가 고정되어 있어 이런 경우를 다루지 못합니다.

In [ ]:
show_quiz("../assets/quizzes/3-4/q32.json")

**정리:** Send는 실행할 때 입력 목록 길이만큼 Worker를 만듭니다. 여러 Worker의 결과는 reducer 같은 병합 규칙으로 모읍니다.

<details><summary>강사용 메모</summary>

- 실습 코드의 `DynamicState.results`는 `Annotated[list[str], lambda a, b: a + b]`입니다. 여러 worker가 같은 `results`에 쓰므로, 두 목록을 이어 붙이는 규칙이 없으면 값이 겹쳐 쓰입니다. 수강생이 물으면 짚어 줍니다.

</details>

## 4. 정리 퀴즈 <small>(퀴즈)</small>

In [ ]:
show_quiz("../assets/quizzes/3-4/final.json")

**이 노트북에서 본 것**
- Subgraph는 따로 컴파일한 그래프를 부모 노드 함수 안에서 호출해 씁니다. State는 따로 정의할 수 있습니다.
- Checkpointer는 같은 `thread_id`의 State를 이어 줍니다. MemorySaver는 메모리에, SqliteSaver는 파일에 저장합니다. 실행 이력을 보고 과거 시점에서 다시 실행할 수도 있습니다.
- Memory Store는 thread를 넘어 남길 사용자 정보를 저장합니다. namespace로 누구의 기억인지 나눕니다.
- Fan-out/Fan-in은 독립적인 작업을 엣지로 나눠 동시에 실행하고 한 노드에서 모읍니다.
- Send는 실행할 때 입력 목록 길이만큼 Worker를 만듭니다. 결과는 reducer 같은 병합 규칙으로 모읍니다.